# OpenAI Agent SDK

1. SDK Tutorial
- Part 1: A simple "Agent" and "Agent Loop"
- Part 2: Adding a Tool
- Part 3: Adding Memory

2. Re-implment Checklist project with SDK

# Part 1: A simple "Agent" and "Agent Loop"

In [1]:
# Imports
import os
import requests
from IPython.display import Markdown, display
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
from rich.console import Console
load_dotenv(override=True)

True

In [2]:
# Create an agent with name, instructions and a model

agent = Agent(name = "Motivator", instructions="You tell motivational Quotes", model="gpt-5.4-mini")

In [3]:
# Run the motivator agent with Runner.run(agent, prompt)

result = await Runner.run(agent, "I am feeling confused")

In [4]:
print(result.final_output)

“When you are confused, don’t rush to have all the answers. Take one small step forward—clarity often comes after movement.”

“Confusion is not failure; it is often the mind’s way of telling you that growth is happening.”

“Even the darkest night ends with sunrise. Trust that this confusing moment will pass.”

“Sometimes the best way to find yourself is to pause, breathe, and keep going gently.”

If you want, I can also give you a few more motivational quotes for confusion, stress, or self-doubt.


In [5]:
result.to_input_list()

[{'content': 'I am feeling confused', 'role': 'user'},
 {'id': 'msg_06f68b9508a06a73006ad9085c8c2887d2acf74f79b4f62667',
  'content': [{'annotations': [],
    'text': '“When you are confused, don’t rush to have all the answers. Take one small step forward—clarity often comes after movement.”\n\n“Confusion is not failure; it is often the mind’s way of telling you that growth is happening.”\n\n“Even the darkest night ends with sunrise. Trust that this confusing moment will pass.”\n\n“Sometimes the best way to find yourself is to pause, breathe, and keep going gently.”\n\nIf you want, I can also give you a few more motivational quotes for confusion, stress, or self-doubt.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

## Adding Observability with a trace

In [6]:
with trace("Telling a motivational quote"):
    result = await Runner.run(agent, "Give me 5 morning motivations")

display(Markdown(result.final_output))

Absolutely — here are 5 morning motivation quotes:

1. **“Every morning is a new opportunity to grow, improve, and become your best self.”**  
2. **“Start your day with a positive thought and a grateful heart.”**  
3. **“The sun rises every morning to remind you that you can rise again.”**  
4. **“Small steps every morning lead to big results over time.”**  
5. **“Believe in yourself today — your future starts with the choices you make this morning.”**

If you want, I can also give you:
- **more powerful morning quotes**
- **short one-line quotes**
- **quotes for work, success, or positivity**

## Let's add streaming

In [7]:
# Streaming
result = Runner.run_streamed(agent, "An athelete is feeling low")

async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush = True)


“Champions are not made by how they feel on their worst day, but by how they rise after it. Keep going—your next breakthrough could be one effort away.”

# Part 2: Adding a tool

In [8]:
pushover_user   = os.getenv("PUSHOVER_USER")
pushover_token  = os.getenv("PUSHOVER_TOKEN")
pushover_url    = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good!")
    else:
        print("Pushover user found, but doesn't look good")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good!")
    else:
        print("Pushover token found and doesn't look good")
else:
    print("Pushover token not found")

Pushover user found and looks good!
Pushover token found and looks good!


In [9]:
@function_tool
def push_tool(message: str) -> str:
    """Send the given message to the user as a push notification"""
    print(f"Push Tool: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, payload)
    return f"Push notification sent with API status code {result}"

In [10]:
push_tool.description

'Send the given message to the user as a push notification'

In [11]:
notifier = Agent(name = "Informer", model="gpt-5.4-mini", instructions="You inform the user upon request", tools=[push_tool])

In [12]:
with trace ("UDP package has arrived"):
    result = await Runner.run(notifier, "Inform the user that the UDP package has arrived")

print(result.final_output)

Push Tool: The UDP package has arrived.
Done — I informed the user that the UDP package has arrived.


# Part 3: Sessions (memory)

### Within a Runner.run() application level turn, the conversation history is maintained. 
- But every Runner.run() call is a fresh start, and it doesn't remember the conversation from previous call
- Let's try

In [13]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [14]:
response = await Runner.run(agent, "Hi, My name is John")
print(response.final_output)

Hi John! Nice to meet you. How can I help you today?


In [15]:
response = await Runner.run(agent, "What is my name?")
print(response.final_output)

I don’t know your name unless you tell me. If you’d like, you can share it and I’ll use it.


### Memory approach 1 - Manually pass in the list of dicts

In [16]:
response = await Runner.run(agent, "Hi, My name is John")
print(response.final_output)

Hi John — nice to meet you! How can I help today?


In [17]:
response.to_input_list()

[{'content': 'Hi, My name is John', 'role': 'user'},
 {'id': 'msg_03c66b86528d3872006ad90868a4ac87d2990ef66195eee53b',
  'content': [{'annotations': [],
    'text': 'Hi John — nice to meet you! How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [18]:
history = response.to_input_list()
next_input = history + [{"role": "user", "content": "What is my name?"}]
next_input


[{'content': 'Hi, My name is John', 'role': 'user'},
 {'id': 'msg_03c66b86528d3872006ad90868a4ac87d2990ef66195eee53b',
  'content': [{'annotations': [],
    'text': 'Hi John — nice to meet you! How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'role': 'user', 'content': 'What is my name?'}]

In [19]:
response = await Runner.run(agent, next_input)
response.final_output

'Your name is John.'

Another appraoch: Use OpenAI Agents SDK's built-in SQLLite session

In [20]:
# This is an in-memory db. It will be stored in RAM and is fast, but not re-usable
# For an on-disk db session use: SQLiteSession("12456", "session_test.db")
session = SQLiteSession("12456")
session

In [21]:
response = await Runner.run(agent, "Hi, my name is Champesh", session=session)
response.final_output

'Hi Champesh — nice to meet you! How can I help you today?'

In [22]:
response = await Runner.run(agent, "What is my name?", session=session)
response.final_output

'Your name is Champesh.'

### Exercise: Checklist Project
- Agent
- Tools: Create checklist, Mark completion

In [13]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completions[index] == True:
            result += f"Checklist #{index+1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index+1}: {item}\n"
    show(result)
    return result


In [14]:
checklist, completions = [], []

@function_tool
def create_checklist(descriptions: list[str]) -> str:
    """Creates a list of checklist items from the given list of descriptions"""
    checklist.extend(descriptions)
    completions.extend([False]*len(descriptions))
    return get_checklist_report()

In [15]:
@function_tool
def mark_completion(index: int, notes: str) -> str:
    """Marks the checklist item as completed based on the index provided. The index starts from 1.
        index: index number of the checklist item.
        notes: one sentence about the progress
    """
    if 1 <= index <= len(completions): 
        completions[index-1] = True
        show(f"{notes}")
    else:
        return "Invalid Index"
    return get_checklist_report()

In [35]:
system_prompt = """
You are a helpful assistant.

If you are given a quants question to solve, using the checklist tools, to setup a checklist, finish and mark each step (one at a time), and reply with a solution.
If the question is missing a quantity, do not ask the user for clarification. Instead, make that a step, and make a reasonable assumption.
Provide your solution without code blocks.""".strip()

In [36]:
tools = [create_checklist, mark_completion]
agent = Agent(name="Assistant", model = "gpt-5.5", instructions=system_prompt,tools = tools)

In [37]:
prompt = """A store offers a 20% discount on a jacket. After the discount, an 8% sales tax is applied. If the original price of the jacket is $150, what is the final price paid by the customer?

A. $120.00
B. $124.80
C. $129.60
D. $135.00
E. $162.00"""

completions, checklist = [], []
session = SQLiteSession("12589")
with trace("Gre problem solving"):
    result = Runner.run_streamed(agent, prompt, session=session)
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush = True)


Checklist #1: Calculate the discounted price after applying the 20% discount to $150.
Checklist #2: Calculate the 8% sales tax on the discounted price.
Checklist #3: Add the sales tax to the discounted price and select the matching answer choice.

A 20% discount on $150 is $30, so the discounted price is $120.

Checklist #1: Calculate the discounted price after applying the 20% discount to $150.
Checklist #2: Calculate the 8% sales tax on the discounted price.
Checklist #3: Add the sales tax to the discounted price and select the matching answer choice.

The 8% sales tax on $120 is $9.60.

Checklist #1: Calculate the discounted price after applying the 20% discount to $150.
Checklist #2: Calculate the 8% sales tax on the discounted price.
Checklist #3: Add the sales tax to the discounted price and select the matching answer choice.

Adding $9.60 tax to the $120 discounted price gives a final price of $129.60.

Checklist #1: Calculate the discounted price after applying the 20% discount to $150.
Checklist #2: Calculate the 8% sales tax on the discounted price.
Checklist #3: Add the sales tax to the discounted price and select the matching answer choice.

Original price: $150

20% discount:
$150 × 0.20 = $30

Discounted price:
$150 − $30 = $120

8% sales tax:
$120 × 0.08 = $9.60

Final price:
$120 + $9.60 = $129.60

Answer: C. $129.60

In [38]:
checklist, completions = [],[]
response = await Runner.run(agent, "Could you explain the framework you used to solve this problem? I would like to understand it and reuse it to solve other questions", session=session)
print(response.final_output)

Sure. The reusable framework is:

1. Identify the order of operations  
   - Discounts are usually applied first.
   - Taxes are usually applied after the discount, unless the problem says otherwise.

2. Convert percentages to decimals  
   - 20% = 0.20
   - 8% = 0.08

3. Apply the discount  
   Formula:  
   Discounted price = Original price × (1 − discount rate)

   In this problem:  
   $150 × (1 − 0.20) = $150 × 0.80 = $120

4. Apply the tax  
   Formula:  
   Final price = Discounted price × (1 + tax rate)

   In this problem:  
   $120 × (1 + 0.08) = $120 × 1.08 = $129.60

5. Match the result to the answer choices  
   $129.60 matches choice C.

Shortcut formula:

Final price = Original price × (1 − discount rate) × (1 + tax rate)

For this problem:

$150 × 0.80 × 1.08 = $129.60

So the final answer is C. $129.60.
